# Mirage · run 2 (resume)

Full suites → pass-rate filter → **both GRPO arms** → audit, unattended.

**How to run:** right panel → Accelerator **GPU T4 x2**, Internet **On**, Input **spider**.
Then **Save Version → Save & Run All (Commit) → Save**. Close the tab; it runs on Kaggle for up to 12 h.
In the morning open the version → **Output** (everything is in `runs/`) and read the last cell (`RUN2.md`).

Time is shared out automatically: each arm gets the same budget and stops cleanly (checkpoint saved) if it runs out.
An unfinished arm resumes from its checkpoint next session.

In [ ]:
# --- setup ---------------------------------------------------------------
import os, subprocess, sys, time, json
T0 = time.time()
SESSION_HOURS = 12.0                       # Kaggle's max for a committed GPU run
SPIDER_DIR = "/kaggle/input/datasets/siddhartharajat/spider/spider_data"
RUNS = "/kaggle/working/runs"
os.environ.update(SPIDER_DIR=SPIDER_DIR, MIRAGE_RUNS=RUNS, CUDA_VISIBLE_DEVICES="0",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
os.makedirs(RUNS, exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/Mirage", "mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])   # Kaggle's torchao 0.10 breaks PEFT
os.chdir("/kaggle/working/mirage")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{SPIDER_DIR}/train_spider.json"), "SPIDER_DIR is wrong: run  !find /kaggle/input -name train_spider.json"
import torch
assert torch.cuda.is_available(), "NO GPU: set Accelerator to GPU T4 x2 in the right panel, then run again"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- helpers ----------------------------------------------------------------
def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600

def sh(cmd, log=None, check=True):
    """Run a shell command, stream its output here, keep a copy in runs/logs."""
    print(f"\n$ {cmd}   [{hours_left():.2f} h left]", flush=True)
    os.makedirs(f"{RUNS}/logs", exist_ok=True)
    lines = []
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    if log:
        open(f"{RUNS}/logs/{log}.log", "w").write("".join(lines))
    if rc and check:
        raise RuntimeError(f"exit {rc}: {cmd}")
    return "".join(lines)

STATUS = {}
def step(name, fn):
    """Run one stage; record success/failure without killing the whole night."""
    t = time.time()
    try:
        out = fn(); STATUS[name] = "ok"
        return out
    except Exception as e:
        STATUS[name] = f"FAILED: {e}"
        print(f"\n!!! {name} failed: {e}")
    finally:
        print(f"[{name}] {(time.time()-t)/60:.1f} min, {hours_left():.2f} h left")

## 0 · Reuse the previous run's work
Attach the earlier run as input: right panel → **Add Input** → **Your Work** → notebook **notebookc1bb2bcff3** (version 2).
This copies its suites, filter result and the single arm's checkpoint, so nothing is recomputed. Without that input, everything runs from scratch.

In [ ]:
import glob, shutil
prev = [p[:-len("/filter/keep.jsonl")] for p in glob.glob("/kaggle/input/**/runs/filter/keep.jsonl", recursive=True)]
if prev:
    src = prev[0]
    print("reusing", src)
    shutil.copytree(src, RUNS, dirs_exist_ok=True)
    for f in glob.glob(f"{RUNS}/logs/*.log"):          # keep the old logs under a different name
        os.rename(f, f.replace(".log", ".v2.log"))
    print("copied:", sorted(os.listdir(RUNS)))
    print("single-arm checkpoints:", sorted(glob.glob(f"{RUNS}/arms/single/checkpoint-*")))
else:
    print("no previous run attached; starting from scratch")

## 1 · Suites for all 166 databases (CPU, ~20–30 min) — and the full blind spot

In [ ]:
if os.path.isdir(f"{RUNS}/suites") and len(os.listdir(f"{RUNS}/suites")) >= 160:
    STATUS["suites"] = "reused"; print("suites reused:", len(os.listdir(f"{RUNS}/suites")), "databases")
else:
  step("suites", lambda: sh(f"mirage suites build --spider {SPIDER_DIR} --out {RUNS}/suites", log="suites"))

## 2 · Pass-rate filter on a hashed subset (GPU, ~2 h)
GRPO sees 300 steps × 8 questions = 2,400 question draws, so filtering all ~7,800 training
questions (~6 h) would waste most of the night. 1,500 questions chosen by hash: spread over all
training databases, identical on re-runs, shared by both arms.

In [ ]:
open("/kaggle/working/filter_subset.py", "w").write('"""Pass-rate filter on a fixed, hashed subset of training questions.\n\nSame logic as `mirage filter`, but on N questions instead of all ~7,800:\nGRPO only sees 300 steps x 8 questions = 2,400 question draws, so filtering\nthe whole training set would spend ~6 GPU hours on questions it never uses.\nThe subset is chosen by hashing each question id, so it is spread across all\ntraining databases, identical on every re-run, and the same for both arms.\n\nusage: python filter_subset.py N [k]\n"""\n\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom mirage.config import load_config, parse_overrides\nfrom mirage.eval.generate import GenConfig, build_generator\nfrom mirage.reward.suite import SuiteIndex\nfrom mirage.reward.verifier import Verifier\nfrom mirage.train.common import resolve_start\nfrom mirage.train.data import load_sets, prompt_spec\nfrom mirage.train.filter import pass_rate_filter\n\nN = int(sys.argv[1])\nK = int(sys.argv[2]) if len(sys.argv) > 2 else 8\nCONFIG = sys.argv[3] if len(sys.argv) > 3 else "configs/grpo_base.yaml"\n\ncfg = load_config(CONFIG, parse_overrides([]))\ncfg.data.filtered_file = None\nsets = load_sets(cfg)\n\n\ndef h(qid: str) -> str:\n    return hashlib.sha256(f"mirage-filter-v1:{qid}".encode()).hexdigest()\n\n\nsubset = sorted(sets.train, key=lambda e: h(e.qid))[:N]\nprint(f"[filter] {len(subset)} of {len(sets.train)} training questions, "\n      f"{len({e.db_id for e in subset})} of {len({e.db_id for e in sets.train})} databases, k={K}", flush=True)\n\ngen = build_generator(GenConfig(backend=cfg.eval.backend, model=resolve_start(cfg), dtype=cfg.model.dtype,\n                                batch_size=cfg.eval.batch_size, chat_template_kwargs=cfg.model.chat_template_kwargs,\n                                base_url=cfg.eval.base_url))\n_gen, _done, _t0 = gen.generate, [0], __import__("time").time()\ndef _progress(prompts, n, temperature):\n    r = _gen(prompts, n, temperature); _done[0] += len(prompts)\n    el = __import__("time").time() - _t0\n    print(f"[filter] {_done[0]}/{len(subset)} questions, {el/60:.1f} min, ~{el/_done[0]*(len(subset)-_done[0])/60:.0f} min left", flush=True)\n    return r\ngen.generate = _progress\nout = Path(cfg.out).parent / "filter" / "keep.jsonl"\nres = pass_rate_filter(subset, gen, Verifier(sets.db_paths(), SuiteIndex(cfg.suites)), prompt_spec(cfg), out, k=K)\nres.update({"subset_of": len(sets.train), "k": K, "subset_rule": "sha256(\'mirage-filter-v1:\'+qid), first N"})\n(out.parent / "summary.json").write_text(json.dumps(res, indent=2))\nprint(json.dumps(res, indent=2))\nprint(f"wrote {out}")\n')
N_FILTER = 1500
if os.path.exists(f"{RUNS}/filter/keep.jsonl"):
    STATUS["filter"] = "reused"; print(open(f"{RUNS}/filter/summary.json").read() if os.path.exists(f"{RUNS}/filter/summary.json") else "filter reused")
else:
    step("filter", lambda: sh(f"python /kaggle/working/filter_subset.py {N_FILTER} 8", log="filter"))

## 3 · Both GRPO arms, equal time budget
Same filtered questions, same seed, same starting weights; only `reward.mode` differs.
Both arms train for the same **300 steps** (seed 0); the single arm resumes from the earlier run's checkpoint. The time budget is only a safety net: if an arm runs out it saves a checkpoint, and we resume it to 200 before comparing.

In [ ]:
STEPS, SEED = 300, 0     # identical for both arms (matches the earlier run's checkpoint); seed 1 later
SUITE_RESERVE = 4.25     # hours kept back for the suite arm (300 steps x ~45 s ~= 3.75 h + overhead)
def arm(name, hours):
    return lambda: sh(f"mirage grpo -c configs/grpo_{name}.yaml --set budget.max_hours={hours:.2f} "
                      f"--set grpo.max_steps={STEPS} --set grpo.seed={SEED}", log=f"grpo_{name}")

if STATUS.get("filter") in ("ok", "reused"):
    h1 = max(0.5, hours_left() - SUITE_RESERVE - 0.5)
    print(f"single arm budget {h1:.2f} h (resumes from its newest checkpoint if one was copied in)")
    step("grpo_single", arm("single", h1))
    h2 = max(0.5, hours_left() - 0.5)
    print(f"suite arm budget {h2:.2f} h")
    step("grpo_suite", arm("suite", h2))
else:
    print("filter failed; skipping training")

## 4 · Audit: how often the training reward lied, and did it grow?

In [ ]:
for a in ["single", "suite"]:
    led = f"{RUNS}/arms/{a}/ledger.jsonl"
    if os.path.exists(led):
        step(f"audit_{a}", lambda led=led, a=a: sh(f"mirage audit {led}", log=f"audit_{a}"))

## 5 · Summary (read this in the morning)

In [ ]:
import glob, re
def tail(path, n=12):
    try: return "".join(open(path).readlines()[-n:])
    except Exception: return "(missing)"
def last_step(a):
    cks = sorted(glob.glob(f"{RUNS}/arms/{a}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    return cks[-1].rsplit("-", 1)[1] if cks else "none"

blind = re.findall(r"blind spot:.*", tail(f"{RUNS}/logs/suites.log", 40))
rows = re.findall(r"\] (\S+): (\d+) gold, (\d+) mutants, (\d+) pass the\s+original, (\d+)\s+caught",
                  open(f"{RUNS}/logs/suites.log").read() if os.path.exists(f"{RUNS}/logs/suites.log") else "")
rows = [(d, int(m), int(p), int(c)) for d, g, m, p, c in rows]
degen = [d for d, m, p, c in rows if m and p == m and c == 0]   # every mutant passes, nothing caught: empty tables
def bs(rs):
    M = sum(r[1] for r in rs); P = sum(r[2] for r in rs); C = sum(r[3] for r in rs)
    return f"{len(rs)} DBs, {M} mutants: original accepts {P/M:.1%}; suites catch {C/max(P,1):.1%} of those; provably wrong-but-rewarded >= {C/M:.1%} of all mutants"
blind_detail = (f"all: {bs(rows)}\nexcluding {len(degen)} degenerate DBs ({', '.join(degen)}): "
                f"{bs([r for r in rows if r[0] not in degen])}") if rows else "(no per-DB lines parsed)"
report = f"""# Run 2 summary

ran {(time.time()-T0)/3600:.2f} h · commit {subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout.strip()}

## Stage status
{json.dumps(STATUS, indent=2)}

## Blind spot (all databases)
{blind[-1] if blind else "(not found, see logs/suites.log)"}
{blind_detail}

## Filter
{tail(f"{RUNS}/filter/summary.json", 20)}

## Training reached
single: step {last_step("single")} · suite: step {last_step("suite")}  (target 300, seed 0; if the two differ, resume the shorter arm before comparing)

## Audit
{tail(f"{RUNS}/logs/audit_single.log", 6)}
{tail(f"{RUNS}/logs/audit_suite.log", 6)}
"""
open(f"{RUNS}/RUN2.md", "w").write(report)
print(report)